In [16]:
import pandas as pd
import numpy as np
import os

def backtest(folder_path):
    # 读取数据
    d = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
    stock = d['kdcode'].unique()

    all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]

    merged_df = pd.DataFrame()

    for file in all_files:
        df = pd.read_csv(file)
        df.rename(columns={'kdcode': 'instrument'}, inplace=True)
        df.rename(columns={'dt': 'datetime'}, inplace=True)
        merged_df = pd.concat([merged_df, df])

    merged_df = merged_df[['datetime', 'instrument', 'score']]
    merged_df['datetime'] = merged_df['datetime'].astype('datetime64[ns]')
    merged_df = merged_df.sort_values(by=['datetime', 'instrument'], ascending=[True, True])
    merged_df = merged_df[merged_df['instrument'].isin(stock)]

    df = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
    df['dt'] = pd.to_datetime(df['dt'])
    df = df[['kdcode', 'dt', 'close']]
    df = df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'})
    df = df.sort_values(by=['instrument', 'datetime'])
    df['close_r'] = df['close'] / df['close'].shift(1)
    df_trade = df
    df_trade = df_trade[df_trade['datetime'] > '2022-12-31']

    x = pd.merge(df_trade, merged_df, on=['datetime', 'instrument'], how='outer')
    trade_date_unique = df_trade['datetime'].unique()
    df_return = pd.DataFrame()

    for date in trade_date_unique:
        x_day = x[x['datetime']==date]
        r_day = x_day.nlargest(10, columns='score').close_r.mean()
        r_day = r_day - 1
        b = {'datetime':date, 'daily_return':r_day}
        df_return = df_return.append(b, ignore_index=True)

    portfolio_df_performance = df_return.set_index(['datetime'])

    alpha_df_performance = pd.DataFrame()
    alpha_df_performance['portfolio_daily_return'] = portfolio_df_performance['daily_return']
    alpha_df_performance['portfolio_net_value'] = (alpha_df_performance['portfolio_daily_return'] + 1).cumprod()

    net_value_columns = ['portfolio_net_value']

    alpha_statistics_df = pd.DataFrame(index=alpha_df_performance[net_value_columns].columns,
                                        columns=["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"])

    alpha_df_performance.index = pd.to_datetime(alpha_df_performance.index)
    monthly_statistics_df = alpha_df_performance[net_value_columns].resample('m').last()
    monthly_statistics_df = pd.concat([alpha_df_performance[:1][['portfolio_net_value']], monthly_statistics_df])
    monthly_statistics_df = monthly_statistics_df.pct_change()
    monthly_statistics_df = monthly_statistics_df.dropna()
    monthly_statistics_df.index = monthly_statistics_df.index.date
    
    yearly_statistics_df = alpha_df_performance[net_value_columns].resample('y').last()
    yearly_statistics_df = pd.concat([alpha_df_performance[:1][['portfolio_net_value']], yearly_statistics_df])
    yearly_statistics_df = yearly_statistics_df.pct_change()
    yearly_statistics_df = yearly_statistics_df.dropna()
    yearly_statistics_df.index = yearly_statistics_df.index.date

    alpha_statistics_df.loc[:, "年化收益"] = np.mean((alpha_df_performance[net_value_columns].tail(1)) ** (252 / len(alpha_df_performance)) - 1)
    alpha_statistics_df.loc[:, "年化波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252)
    alpha_statistics_df.loc[:, "累积收益"] = np.mean(alpha_df_performance[net_value_columns].tail(1) - 1)
    alpha_statistics_df.loc[:, "累积波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)
    alpha_statistics_df.loc[:, "最大回撤率"] = np.min((alpha_df_performance[net_value_columns] - alpha_df_performance[net_value_columns].cummax()) / alpha_df_performance[net_value_columns].cummax())
    alpha_statistics_df.loc[:, "夏普率"] = alpha_statistics_df["年化收益"] / alpha_statistics_df["年化波动率"]
    alpha_statistics_df.loc[:, "Calmar"] = alpha_statistics_df["年化收益"] / abs(alpha_statistics_df["最大回撤率"])
    alpha_statistics_df.loc[:, "IR"] = np.mean(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252) / np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)
    
    return alpha_statistics_df

results = []
base_path = '/home/liyuante/neruocomputing/20240621_hs300_0.5_5_10_2_10/prediction_0'
for i in range(10):
    folder_path = os.path.join(base_path, str(i))
    result = backtest(folder_path)
    results.append(result)

for i, result in enumerate(results):
    print(result)
    print()


Folder 0:
                        年化收益     年化波动率     最大回撤率       夏普率   Calmar        IR  \
portfolio_net_value -0.05462  0.176187 -0.215737 -0.310013 -0.25318 -0.409033   

                         累积收益     累积波动率  
portfolio_net_value -0.052511  0.011099  

Folder 1:
                         年化收益     年化波动率    最大回撤率       夏普率    Calmar  \
portfolio_net_value -0.101556  0.171338 -0.23748 -0.592721 -0.427638   

                           IR      累积收益     累积波动率  
portfolio_net_value -0.636994 -0.097729  0.010793  

Folder 2:
                         年化收益     年化波动率     最大回撤率       夏普率    Calmar  \
portfolio_net_value -0.067604  0.172697 -0.249197 -0.391459 -0.271287   

                           IR     累积收益     累积波动率  
portfolio_net_value -0.431112 -0.06501  0.010879  

Folder 3:
                         年化收益     年化波动率     最大回撤率       夏普率    Calmar  \
portfolio_net_value  0.052005  0.171505 -0.169142  0.303228  0.307464   

                           IR      累积收益     累积波动率  
portfolio_net

In [19]:
results[0]

,年化收益,年化波动率,最大回撤率,夏普率,Calmar,IR,累积收益,累积波动率
portfolio_net_value,-0.05462,0.176187,-0.215737,-0.310013,-0.25318,-0.409033,-0.052511,0.011099


In [20]:
# 显示结果
final_df = pd.concat(results)
print(final_df)

                         年化收益     年化波动率     最大回撤率       夏普率    Calmar  \
portfolio_net_value -0.054620  0.176187 -0.215737 -0.310013 -0.253180   
portfolio_net_value -0.101556  0.171338 -0.237480 -0.592721 -0.427638   
portfolio_net_value -0.067604  0.172697 -0.249197 -0.391459 -0.271287   
portfolio_net_value  0.052005  0.171505 -0.169142  0.303228  0.307464   
portfolio_net_value  0.030800  0.175650 -0.191674  0.175347  0.160687   
portfolio_net_value -0.107138  0.172627 -0.215234 -0.620633 -0.497775   
portfolio_net_value -0.080099  0.171494 -0.244066 -0.467069 -0.328188   
portfolio_net_value  0.019441  0.173280 -0.160526  0.112192  0.121106   
portfolio_net_value -0.087552  0.165586 -0.233774 -0.528738 -0.374514   
portfolio_net_value -0.101659  0.171586 -0.239770 -0.592469 -0.423987   

                           IR      累积收益     累积波动率  
portfolio_net_value -0.409033 -0.052511  0.011099  
portfolio_net_value -0.636994 -0.097729  0.010793  
portfolio_net_value -0.431112 -0.065010 

In [7]:
import pandas as pd
import numpy as np
import os

def backtest(folder_path):
    # 读取数据
    d = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
    stock = d['kdcode'].unique()

    all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]

    merged_df = pd.DataFrame()

    for file in all_files:
        df = pd.read_csv(file)
        df.rename(columns={'kdcode': 'instrument'}, inplace=True)
        df.rename(columns={'dt': 'datetime'}, inplace=True)
        merged_df = pd.concat([merged_df, df])

    merged_df = merged_df[['datetime', 'instrument', 'score']]
    merged_df['datetime'] = merged_df['datetime'].astype('datetime64[ns]')
    merged_df = merged_df.sort_values(by=['datetime', 'instrument'], ascending=[True, True])
    merged_df = merged_df[merged_df['instrument'].isin(stock)]

    df = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
    df['dt'] = pd.to_datetime(df['dt'])
    df = df[['kdcode', 'dt', 'close']]
    df = df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'})
    df = df.sort_values(by=['instrument', 'datetime'])
    df['close_r'] = df['close'] / df['close'].shift(1)
    df_trade = df
    df_trade = df_trade[df_trade['datetime'] > '2022-12-31']

    x = pd.merge(df_trade, merged_df, on=['datetime', 'instrument'], how='outer')
    trade_date_unique = df_trade['datetime'].unique()
    df_return = pd.DataFrame()

    for date in trade_date_unique:
        x_day = x[x['datetime']==date]
        r_day = x_day.nlargest(10, columns='score').close_r.mean()
        r_day = r_day - 1
        b = {'datetime':date, 'daily_return':r_day}
        df_return = df_return.append(b, ignore_index=True)

    portfolio_df_performance = df_return.set_index(['datetime'])

    alpha_df_performance = pd.DataFrame()
    alpha_df_performance['portfolio_daily_return'] = portfolio_df_performance['daily_return']
    alpha_df_performance['portfolio_net_value'] = (alpha_df_performance['portfolio_daily_return'] + 1).cumprod()

    net_value_columns = ['portfolio_net_value']

    alpha_statistics_df = pd.DataFrame(index=alpha_df_performance[net_value_columns].columns,
                                        columns=["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"])

    alpha_df_performance.index = pd.to_datetime(alpha_df_performance.index)
    monthly_statistics_df = alpha_df_performance[net_value_columns].resample('m').last()
    monthly_statistics_df = pd.concat([alpha_df_performance[:1][['portfolio_net_value']], monthly_statistics_df])
    monthly_statistics_df = monthly_statistics_df.pct_change()
    monthly_statistics_df = monthly_statistics_df.dropna()
    monthly_statistics_df.index = monthly_statistics_df.index.date
    
    yearly_statistics_df = alpha_df_performance[net_value_columns].resample('y').last()
    yearly_statistics_df = pd.concat([alpha_df_performance[:1][['portfolio_net_value']], yearly_statistics_df])
    yearly_statistics_df = yearly_statistics_df.pct_change()
    yearly_statistics_df = yearly_statistics_df.dropna()
    yearly_statistics_df.index = yearly_statistics_df.index.date

    alpha_statistics_df.loc[:, "年化收益"] = np.mean((alpha_df_performance[net_value_columns].tail(1)) ** (252 / len(alpha_df_performance)) - 1)
    alpha_statistics_df.loc[:, "年化波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252)
    alpha_statistics_df.loc[:, "累积收益"] = np.mean(alpha_df_performance[net_value_columns].tail(1) - 1)
    alpha_statistics_df.loc[:, "累积波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)
    alpha_statistics_df.loc[:, "最大回撤率"] = np.min((alpha_df_performance[net_value_columns] - alpha_df_performance[net_value_columns].cummax()) / alpha_df_performance[net_value_columns].cummax())
    alpha_statistics_df.loc[:, "夏普率"] = alpha_statistics_df["年化收益"] / alpha_statistics_df["年化波动率"]
    alpha_statistics_df.loc[:, "Calmar"] = alpha_statistics_df["年化收益"] / abs(alpha_statistics_df["最大回撤率"])
    alpha_statistics_df.loc[:, "IR"] = np.mean(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252) / np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)
    
    return alpha_statistics_df

# 主程序
results = []
base_path = '/home/liyuante/neruocomputing/20240621_hs300_0.8_5_10_5_10/prediction_4'
for i in range(10):
    folder_path = os.path.join(base_path, str(i))
    result = backtest(folder_path)
    result = result[["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"]]  # 只保留6个指标
    result['folder'] = i
    results.append(result)

# 显示结果
final_df = pd.concat(results).reset_index(drop=True)
print(final_df)

       年化收益     年化波动率     最大回撤率       夏普率    Calmar        IR  folder
0  0.335049  0.174137 -0.123653  1.924047  2.709579  1.729803       0
1  0.510638  0.207357 -0.122884  2.462606  4.155435  2.063661       1
2  0.419579  0.215530 -0.139072  1.946732  3.016996  1.738824       2
3  0.320444  0.224445 -0.154223  1.427715  2.077791  1.289207       3
4  0.237747  0.186787 -0.127529  1.272822  1.864255  1.257541       4
5  0.350093  0.195633 -0.156449  1.789541  2.237747  1.674884       5
6  0.429508  0.213174 -0.112595  2.014828  3.814623  1.828003       6
7  0.339600  0.217763 -0.111918  1.559499  3.034377  1.529116       7
8  0.525886  0.211244 -0.122924  2.489473  4.278148  2.015646       8
9  0.171869  0.204290 -0.126120  0.841301  1.362746  0.924543       9


In [1]:
import pandas as pd
import numpy as np
import os

def backtest(average_scores_df, stock):
    average_scores_df = average_scores_df[['datetime', 'instrument', 'score']]
    average_scores_df['datetime'] = average_scores_df['datetime'].astype('datetime64[ns]')
    average_scores_df = average_scores_df.sort_values(by=['datetime', 'instrument'], ascending=[True, True])
    average_scores_df = average_scores_df[average_scores_df['instrument'].isin(stock)]

    stock_df = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
    stock_df['dt'] = pd.to_datetime(stock_df['dt'])
    stock_df = stock_df[['kdcode', 'dt', 'close']]
    stock_df = stock_df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'})
    stock_df = stock_df.sort_values(by=['instrument', 'datetime'])
    stock_df['close_r'] = stock_df['close'] / stock_df['close'].shift(1)
    df_trade = stock_df[stock_df['datetime'] > '2022-12-31']

    x = pd.merge(df_trade, average_scores_df, on=['datetime', 'instrument'], how='outer')
    trade_date_unique = df_trade['datetime'].unique()
    df_return = pd.DataFrame()

    for date in trade_date_unique:
        x_day = x[x['datetime'] == date]
        r_day = x_day.nlargest(10, columns='score').close_r.mean()
        r_day = r_day - 1
        b = {'datetime': date, 'daily_return': r_day}
        df_return = df_return.append(b, ignore_index=True)

    portfolio_df_performance = df_return.set_index(['datetime'])

    alpha_df_performance = pd.DataFrame()
    alpha_df_performance['portfolio_daily_return'] = portfolio_df_performance['daily_return']
    alpha_df_performance['portfolio_net_value'] = (alpha_df_performance['portfolio_daily_return'] + 1).cumprod()

    net_value_columns = ['portfolio_net_value']

    alpha_statistics_df = pd.DataFrame(index=alpha_df_performance[net_value_columns].columns,
                                        columns=["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"])

    alpha_df_performance.index = pd.to_datetime(alpha_df_performance.index)
    alpha_statistics_df.loc[:, "年化收益"] = np.mean((alpha_df_performance[net_value_columns].tail(1)) ** (252 / len(alpha_df_performance)) - 1)
    alpha_statistics_df.loc[:, "年化波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252)
    alpha_statistics_df.loc[:, "最大回撤率"] = np.min((alpha_df_performance[net_value_columns] - alpha_df_performance[net_value_columns].cummax()) / alpha_df_performance[net_value_columns].cummax())
    alpha_statistics_df.loc[:, "夏普率"] = alpha_statistics_df["年化收益"] / alpha_statistics_df["年化波动率"]
    alpha_statistics_df.loc[:, "Calmar"] = alpha_statistics_df["年化收益"] / abs(alpha_statistics_df["最大回撤率"])
    alpha_statistics_df.loc[:, "IR"] = np.mean(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252) / np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)

    return alpha_statistics_df

def get_merged_score_df(base_path, prediction_index):
    score_dfs = []
    prediction_path = os.path.join(base_path, f'prediction_{prediction_index}')
    
    for j in range(20):
        folder_path = os.path.join(prediction_path, str(j))
        all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
        
        for file in all_files:
            df = pd.read_csv(file)
            score_dfs.append(df)
    
    # 合并所有score_dfs并取均值
    merged_df = pd.concat(score_dfs).groupby(['dt', 'kdcode']).mean().reset_index()
    merged_df.rename(columns={'kdcode': 'instrument', 'dt' : 'datetime'}, inplace=True)
    return merged_df

# 主程序
base_path = '/home/liyuante/20240702_csi300_0.9_5_10_20_20'
d = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
stock = d['kdcode'].unique()

results = []

for i in range(20):
    average_scores_df = get_merged_score_df(base_path, i)
    result = backtest(average_scores_df, stock)
    result = result[["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"]]  # 只保留6个指标
    result['folder'] = i
    results.append(result)

# 显示结果
final_df = pd.concat(results).reset_index(drop=True)
print(final_df)


        年化收益     年化波动率     最大回撤率       夏普率    Calmar        IR  folder
0  -0.070595  0.164575 -0.189630 -0.428954 -0.372277 -0.489760       0
1   0.038415  0.163269 -0.146741  0.235286  0.261787  0.218177       1
2  -0.045408  0.171123 -0.169343 -0.265351 -0.268140 -0.141935       2
3  -0.046236  0.173348 -0.230013 -0.266723 -0.201015 -0.238073       3
4  -0.107272  0.175758 -0.245323 -0.610339 -0.437269 -0.617112       4
5  -0.181494  0.179778 -0.336416 -1.009543 -0.539491 -1.040864       5
6  -0.085891  0.177256 -0.242205 -0.484562 -0.354622 -0.568733       6
7  -0.174568  0.176401 -0.283257 -0.989609 -0.616289 -1.052976       7
8  -0.079958  0.169002 -0.206175 -0.473119 -0.387815 -0.389283       8
9  -0.157521  0.167512 -0.247023 -0.940355 -0.637677 -0.965188       9
10 -0.028586  0.165522 -0.151697 -0.172703 -0.188442 -0.017658      10
11 -0.140153  0.164803 -0.227951 -0.850430 -0.614840 -0.876008      11
12 -0.208326  0.162444 -0.299822 -1.282448 -0.694832 -1.385506      12
13 -0.

In [1]:
import pandas as pd
import numpy as np
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def backtest(average_scores_df, stock):
    average_scores_df = average_scores_df[['datetime', 'instrument', 'score']]
    average_scores_df['datetime'] = average_scores_df['datetime'].astype('datetime64[ns]')
    average_scores_df = average_scores_df.sort_values(by=['datetime', 'instrument'], ascending=[True, True])
    average_scores_df = average_scores_df[average_scores_df['instrument'].isin(stock)]

    stock_df = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
    stock_df['dt'] = pd.to_datetime(stock_df['dt'])
    stock_df = stock_df[['kdcode', 'dt', 'close']]
    stock_df = stock_df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'})
    stock_df = stock_df.sort_values(by=['instrument', 'datetime'])
    stock_df['close_r'] = stock_df['close'] / stock_df['close'].shift(1)
    df_trade = stock_df[stock_df['datetime'] > '2022-12-31']

    x = pd.merge(df_trade, average_scores_df, on=['datetime', 'instrument'], how='outer')
    trade_date_unique = df_trade['datetime'].unique()
    df_return = pd.DataFrame()

    for date in trade_date_unique:
        x_day = x[x['datetime'] == date]
        r_day = x_day.nlargest(10, columns='score').close_r.mean()
        r_day = r_day - 1
        b = {'datetime': date, 'daily_return': r_day}
        df_return = df_return.append(b, ignore_index=True)

    portfolio_df_performance = df_return.set_index(['datetime'])

    alpha_df_performance = pd.DataFrame()
    alpha_df_performance['portfolio_daily_return'] = portfolio_df_performance['daily_return']
    alpha_df_performance['portfolio_net_value'] = (alpha_df_performance['portfolio_daily_return'] + 1).cumprod()

    net_value_columns = ['portfolio_net_value']

    alpha_statistics_df = pd.DataFrame(index=alpha_df_performance[net_value_columns].columns,
                                       columns=["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"])

    alpha_df_performance.index = pd.to_datetime(alpha_df_performance.index)
    alpha_statistics_df.loc[:, "年化收益"] = np.mean((alpha_df_performance[net_value_columns].tail(1)) ** (252 / len(alpha_df_performance)) - 1)
    alpha_statistics_df.loc[:, "年化波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252)
    alpha_statistics_df.loc[:, "最大回撤率"] = np.min((alpha_df_performance[net_value_columns] - alpha_df_performance[net_value_columns].cummax()) / alpha_df_performance[net_value_columns].cummax())
    alpha_statistics_df.loc[:, "夏普率"] = alpha_statistics_df["年化收益"] / alpha_statistics_df["年化波动率"]
    alpha_statistics_df.loc[:, "Calmar"] = alpha_statistics_df["年化收益"] / abs(alpha_statistics_df["最大回撤率"])
    alpha_statistics_df.loc[:, "IR"] = np.mean(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252) / np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)

    return alpha_statistics_df

def get_merged_score_df(base_path, prediction_index):
    score_dfs = []
    prediction_path = os.path.join(base_path, f'prediction_{prediction_index}')
    
    for j in range(20):
        folder_path = os.path.join(prediction_path, str(j))
        all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
        
        for file in all_files:
            df = pd.read_csv(file)
            score_dfs.append(df)
    
    # 合并所有score_dfs并取均值
    merged_df = pd.concat(score_dfs).groupby(['dt', 'kdcode']).mean().reset_index()
    merged_df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'}, inplace=True)
    return merged_df

def process_folder(base_path, stock, index):
    average_scores_df = get_merged_score_df(base_path, index)
    result = backtest(average_scores_df, stock)
    result = result[["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"]]  # 只保留6个指标
    result['folder'] = index
    return result

# 主程序
base_path = '/home/liyuante/20240703_csi300_0.8_5_6_20_20'
d = pd.read_csv('../dataset/hs300_2018_2023_new_1.csv')
stock = d['kdcode'].unique()

results = []

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(process_folder, base_path, stock, i) for i in range(20)]
    for future in tqdm(futures):
        results.append(future.result())

# 显示结果
final_df = pd.concat(results).reset_index(drop=True)
print(final_df)


100%|██████████| 20/20 [05:25<00:00, 16.29s/it]

        年化收益     年化波动率     最大回撤率       夏普率    Calmar        IR  folder
0   0.615723  0.208768 -0.105450  2.949319  5.839030  2.470333       0
1   0.296445  0.199100 -0.092964  1.488926  3.188816  1.445478       1
2   0.352189  0.225219 -0.151516  1.563762  2.324439  1.541512       2
3   0.312525  0.208639 -0.106638  1.497923  2.930703  1.436465       3
4   0.467256  0.207700 -0.127749  2.249675  3.657611  1.916190       4
5   0.427608  0.208242 -0.161817  2.053415  2.642544  1.779416       5
6   0.161684  0.216090 -0.149728  0.748224  1.079851  0.893002       6
7   0.358105  0.217107 -0.141405  1.649443  2.532481  1.606088       7
8   0.234205  0.209261 -0.232402  1.119203  1.007758  1.072483       8
9   0.645983  0.214211 -0.098850  3.015631  6.534981  2.497458       9
10  0.323060  0.202638 -0.123028  1.594267  2.625898  1.571630      10
11  0.610648  0.230873 -0.094943  2.644949  6.431728  2.273204      11
12  0.213846  0.196963 -0.111622  1.085715  1.915805  1.184104      12
13  0.